### Data preprocessing

In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
from collections import Counter
warnings.filterwarnings('ignore')

base_dir = os.getcwd()
parent_dir = os.path.dirname(base_dir)
os.chdir(parent_dir)

np.random.seed(101)

print(f"Current Directory: {os.getcwd()}")

Current Directory: /Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study


'/Users/guanqiaofeng/Documents/BHK/XevaDB/Manuscript_gitrepo/XevaDB_case_study/2_drugSensitivity'

### Load in expert rating

In [2]:
expert1 = pd.read_json("data/rawdata/pdxe/expert_rating/pdx_results_Guanqiao.json")
expert2 = pd.read_json("data/rawdata/pdxe/expert_rating/pdx_results_Mitchell Elliott.json")
expert3 = pd.read_json("data/rawdata/pdxe/expert_rating/pdx_results_Samah.json")

### Merge expert rating

In [3]:
df1 = pd.DataFrame(expert1['ratings']).reset_index()
df1.columns = ['figure_id', 'rating_guanqiao']

df2 = pd.DataFrame(expert2['ratings']).reset_index()
df2.columns = ['figure_id', 'rating_mitchell']

df3 = pd.DataFrame(expert3['ratings']).reset_index()
df3.columns = ['figure_id', 'rating_samah']

merged_df = pd.merge(df1, df2, on='figure_id', how='outer')
merged_df = pd.merge(merged_df, df3, on='figure_id', how='outer')

merged_df = merged_df.sort_values('figure_id')

merged_df

,figure_id,rating_guanqiao,rating_mitchell,rating_samah
0,fig_000001,cat3,cat3,cat3
1,fig_000002,cat1,cat3,cat1
2,fig_000003,cat2,cat2,cat1
3,fig_000004,cat5,cat4,cat5
4,fig_000005,cat4,cat5,cat4
...,...,...,...,...
1295,fig_001296,cat2,cat2,cat2
1296,fig_001297,cat2,cat2,cat2
1297,fig_001298,cat1,cat1,cat1
1298,fig_001299,cat3,cat3,cat3


### Consolidate rating by majority vote

In [ ]:
def consolidate_ratings(row):
    # Get values and drop 'nan' or None values
    ratings = [row['rating_guanqiao'], row['rating_mitchell'], row['rating_samah']]
    valid_ratings = [str(r) for r in ratings if pd.notna(r)]
    
    if not valid_ratings:
        return "No Ratings"
    
    counts = Counter(valid_ratings)
    most_common = counts.most_common() # Returns list of (value, count)
    
    # most_common[0][1] is the highest frequency found
    if most_common[0][1] >= 2:
        return most_common[0][0]  # Return the majority (2 or 3 votes)
    else:
        # It's a 3-way tie (1, 1, 1) or only 1 person rated
        return "/".join(valid_ratings)
    
merged_df['rating_merge'] = merged_df.apply(consolidate_ratings, axis=1)

### exam data balance

In [8]:
merged_df['rating_merge'].value_counts()

rating_merge
cat2              359
cat1              305
cat3              299
cat4              213
cat5               91
cat2/cat3/cat1      7
cat4/cat3/cat5      3
cat1/cat2           2
cat5/cat3/cat1      2
cat5/cat4/cat1      2
cat3/cat4/cat1      2
cat3/cat2/cat4      2
cat4/cat1/cat3      2
cat2/cat3/cat4      2
cat2/cat1           2
cat3/cat2/cat1      2
cat4/cat2/cat3      1
cat3/cat1           1
cat4/cat1           1
cat4/cat2/cat1      1
cat4/cat3/cat1      1
Name: count, dtype: int64

### exam agreement of experts

In [9]:
# Calculate agreement counts
agree_guanqiao = (merged_df['rating_guanqiao'] == merged_df['rating_merge']).sum()
agree_mitchell = (merged_df['rating_mitchell'] == merged_df['rating_merge']).sum()
agree_samah    = (merged_df['rating_samah'] == merged_df['rating_merge']).sum()

# Total number of unique figures for context
total_figures = len(merged_df)

print(f"Total Figures: {total_figures}")
print("-" * 30)
print(f"Guanqiao matches merge: {agree_guanqiao} ({agree_guanqiao/total_figures:.1%})")
print(f"Mitchell matches merge: {agree_mitchell} ({agree_mitchell/total_figures:.1%})")
print(f"Samah matches merge:    {agree_samah} ({agree_samah/total_figures:.1%})")

Total Figures: 1300
------------------------------
Guanqiao matches merge: 1120 (86.2%)
Mitchell matches merge: 1003 (77.2%)
Samah matches merge:    1098 (84.5%)


### exam consistent vs different

In [ ]:
# We use na=False to handle any potential missing values safely
is_different = merged_df['rating_merge'].str.contains('/', na=False)

# 2. Get the counts
num_different = is_different.sum()
num_consistent = (~is_different).sum()  # Everything that doesn't have a "/"
total_figures = len(merged_df)

# 3. Calculate percentages
pct_different = (num_different / total_figures) * 100
pct_consistent = (num_consistent / total_figures) * 100

# Display results
print("Summary based on 'rating_merge':")
print(f"Total:      {total_figures}")
print(f"Consistent among at least two experts: {num_consistent} ({pct_consistent:.2f}%)")
print(f"Different among all three experts:  {num_different} ({pct_different:.2f}%)")

Summary based on 'rating_merge':
Total:      1300
Consistent among at least two experts: 1267 (97.46%)
Different among all three experts:  33 (2.54%)


### save consolidated rating (ground truth) 

In [ ]:
merged_df.to_csv("data/procdata/2_drugSensitivity/expert_rating/expert_rating_summary.csv", index=False)